***Phase 6 - Feature Engineering***

Business Objective

Create meaningful features that capture seasonality, customer behavior, promotions, competition effects, and temporal patterns to improve sales forecasting.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

df["Date"] = pd.to_datetime(df["Date"])

C:\Users\mahip\AppData\Local\Temp\ipykernel_17288\1326193246.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/cleaned_data.csv")


In [3]:
df = df.sort_values(["Store", "Date"])

In [4]:
df.reset_index(drop=True, inplace=True)

**Date Features**

In [5]:
df["Year"] = df["Date"].dt.year

In [6]:
df["Month"] = df["Date"].dt.month

In [7]:
df["Quarter"] = df["Date"].dt.quarter

In [8]:
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

In [9]:
df["Day"] = df["Date"].dt.day

In [10]:
df["DayOfYear"] = df["Date"].dt.dayofyear

In [11]:
df["IsWeekend"] = (
    df["DayOfWeek"] >= 6
).astype(int)

In [12]:
df["MonthStart"] = (
    df["Date"].dt.is_month_start
).astype(int)

In [13]:
df["MonthEnd"] = (
    df["Date"].dt.is_month_end
).astype(int)

Competition Age

First replace zeros with NaN so they don't produce invalid dates:

In [14]:
comp_year = df["CompetitionOpenSinceYear"].replace(0, np.nan)
comp_month = df["CompetitionOpenSinceMonth"].replace(0, np.nan)

competition_date = pd.to_datetime(
    dict(
        year=comp_year,
        month=comp_month,
        day=1
    ),
    errors="coerce"
)

df["CompetitionAgeMonths"] = (
    (df["Date"] - competition_date).dt.days / 30
).clip(lower=0).fillna(0)

Stores facing competition for years behave differently than stores with new competitors.

Promo Active

Already available.

Convert to integer.

In [16]:
df["Promo"] = df["Promo"].astype(int)

In [17]:
df["Promo2"] = df["Promo2"].astype(int)

**Lag Features**

These are among the strongest predictors.

Since sales yesterday strongly influence today.

In [18]:
df["Sales_Lag_1"] = (
    df.groupby("Store")["Sales"]
      .shift(1)
)

In [19]:
df["Sales_Lag_7"] = (
    df.groupby("Store")["Sales"]
      .shift(7)
)

In [20]:
df["Sales_Lag_14"] = (
    df.groupby("Store")["Sales"]
      .shift(14)
)

**Rolling Mean (7 Days)**

In [21]:
df["RollingMean7"] = (
    df.groupby("Store")["Sales"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

**Rolling Mean (30 Days)**

In [22]:
df["RollingMean30"] = (
    df.groupby("Store")["Sales"]
      .transform(lambda x: x.shift(1).rolling(30).mean())
)

**Rolling Standard Deviation**

In [23]:
df["RollingStd7"] = (
    df.groupby("Store")["Sales"]
      .transform(lambda x: x.shift(1).rolling(7).std())
)

Previous Day Customers

In [24]:
df["Customers_Lag1"] = (
    df.groupby("Store")["Customers"]
      .shift(1)
)

In [25]:
df["CustomerRolling7"] = (
    df.groupby("Store")["Customers"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

**Average Store Sales**

In [26]:
store_avg = (
    df.groupby("Store")["Sales"]
      .mean()
)

df["StoreAverageSales"] = (
    df["Store"]
      .map(store_avg)
)

Average Customers

In [27]:
store_customers = (
    df.groupby("Store")["Customers"]
      .mean()
)

df["StoreAverageCustomers"] = (
    df["Store"]
      .map(store_customers)
)

**Sales Per Customer**

In [28]:
df["SalesPerCustomer"] = (
    df["Sales"] /
    df["Customers"]
)

df["SalesPerCustomer"] = (
    df["SalesPerCustomer"]
      .replace([np.inf, -np.inf], np.nan)
)

**Cyclical Encoding**

Months are cyclical.

December is close to January.

Instead of:

12 → 1

Encode as:

In [29]:
df["MonthSin"] = np.sin(
    2 * np.pi * df["Month"] / 12
)

df["MonthCos"] = np.cos(
    2 * np.pi * df["Month"] / 12
)

In [30]:
df["WeekdaySin"] = np.sin(
    2 * np.pi * df["DayOfWeek"] / 7
)

df["WeekdayCos"] = np.cos(
    2 * np.pi * df["DayOfWeek"] / 7
)

**Handle New Missing Values**

In [31]:
df.isnull().sum().sort_values(ascending=False)

PromoInterval                508031
SalesPerCustomer             172869
RollingMean30                 33450
Sales_Lag_14                  15610
Sales_Lag_7                    7805
CustomerRolling7               7805
RollingMean7                   7805
RollingStd7                    7805
Sales_Lag_1                    1115
Customers_Lag1                 1115
Store                             0
Assortment                        0
StoreType                         0
SchoolHoliday                     0
StateHoliday                      0
Promo                             0
Open                              0
Customers                         0
Sales                             0
Date                              0
DayOfWeek                         0
CompetitionOpenSinceYear          0
CompetitionDistance               0
Promo2                            0
Promo2SinceWeek                   0
Promo2SinceYear                   0
CompetitionOpenSinceMonth         0
MonthEnd                    

In [32]:
lag_cols = [
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "RollingMean7",
    "RollingMean30",
    "RollingStd7",
    "Customers_Lag1",
    "CustomerRolling7"
]

df[lag_cols] = df[lag_cols].fillna(0)

In [33]:
df.isnull().sum().sort_values(ascending=False)

PromoInterval                508031
SalesPerCustomer             172869
Store                             0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
DayOfWeek                         0
Date                              0
Assortment                        0
StoreType                         0
CompetitionOpenSinceYear          0
CompetitionDistance               0
Promo2                            0
Promo2SinceWeek                   0
Promo2SinceYear                   0
CompetitionOpenSinceMonth         0
Month                             0
Quarter                           0
WeekOfYear                        0
Day                               0
DayOfYear                         0
IsWeekend                         0
MonthStart                        0
Year                              0
MonthEnd                    

In [34]:
df["PromoInterval"].value_counts(dropna=False)

PromoInterval
NaN                 508031
Jan,Apr,Jul,Oct     293122
Feb,May,Aug,Nov     118596
Mar,Jun,Sept,Dec     97460
Name: count, dtype: int64

In [35]:
df["PromoInterval"] = df["PromoInterval"].fillna("None")

In [36]:
df.drop(columns=["SalesPerCustomer"], inplace=True)

In [37]:
df.isnull().sum().sort_values(ascending=False)

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
Year                         0
Month                        0
Quarter                      0
WeekOfYear                   0
Day                          0
DayOfYear                    0
IsWeekend                    0
MonthStart                   0
MonthEnd                     0
CompetitionAgeMonths         0
Sales_Lag_1                  0
Sales_Lag_7                  0
Sales_Lag_14                 0
RollingMean7                 0
RollingM

In [39]:
df.to_csv(
    "../data/processed/featured_data.csv",
    index=False
)